In [ ]:
import pandas as pd

import dotenv
dotenv.load_dotenv()

True

# Generating variations of questions given reference questions
## Setting up connections

In [4]:
USE_LANGCHAIN = False  # Set to True for LangChain, False for native Vertex AIi

if USE_LANGCHAIN:
    from langchain_google_vertexai import ChatVertexAI
    llm = ChatVertexAI(
    model_name="gemini-2.5-pro",
    project="gcp-genai-4855-dev-afef")

else:
    import vertexai
    from vertexai.generative_models import GenerativeModel 
    import os

    # Remove any GOOGLE_APPLICATION_CREDENTIALS that might point to missing file
    if 'GOOGLE_APPLICATION_CREDENTIALS' in os.environ:
        del os.environ['GOOGLE_APPLICATION_CREDENTIALS']
    vertexai.init(project="gcp-genai-4855-dev-afef", location="us-central1")
    
    llm = GenerativeModel("gemini-2.5-pro")#("gemini-2.0-flash-exp")


## Loading reference_questions

In [ ]:
reference_questions_df = pd.read_csv('reference_test_questions.csv')

for i,row in reference_questions_df.iterrows():
    assert row['reference_question'] == row['variation_question'], row
    assert row['note'] != '',   row
    assert row['variation_label'] == 0, row
    assert row['tool_calls_complete']==row['tool_calls_complete'],  row 


reference_questions = reference_questions_df['reference_question'].tolist()
for question in reference_questions:
    print(question)
print(len(reference_questions))
    

What is the SMILES of {{molecule_name}}?
What is the SMARTS of {{molecule_name}}?
Can you generate similar molecules to {{smiles}}?
How can I synthesize this molecule: {{smiles}}?
What are similar reactions to {{reaction}}?
How many F and N does {{smiles}} have?
How many hydrogen bond donors and acceptors does {{smiles}} have?
What is the molecular weight and number of rotatable bonds of molecule: {{smiles}}?
How can I synthesize: {{molecule_name}}?
How can I synthesize: {{molecule_name}} and what are some similar reactions to each reaction step involved in making this molecule?
Can you generate similar molecules to: {{molecule_name}}?
I would like to have some close analogues to {{smiles}} with {{property_constraints}}
I would like to have some close analogues to {{molecule_name}} with {{property_constraints}}
Can you generate molecules similar to {{smiles}} and specify their molecular weight and number of rotatable bonds for each molecule? Sort the list according to similarity to the

## Defining the prompts

In [10]:
# Choose which LLM interface to use

system_prompt = """You are an expert chemist working with a helpful assistant with knowledge of chemistry.
            Given a provided question, generate 4 variations of the provided question. When you generate the variations, it is very important to follow the following instructions:
            1. The original meaning should be preserved and clearly conveyed in all variations.
            2. The variations should impersonate how a human scientist would ask the question to an assistant.
            3. When generating the variations, make sure that the variations have different levels of formality, from very formal to very informal. The informal versions could even contain some abbrevations, mistakes or spelling errors.
            4. Anything that is encapsulated in {{}} is a variable and should be kept as is in the variations. Do not replace the variable with any value.
            5. If the questions is asking for a specific number of items, make sure to keep that number in each variation.
            6. If the question is asking for a specific property, make sure to keep the exact same property in each variation.
            7. If the question is asking to generate molecules make sure to keep the meaning the same in each variation, and do not change it to something like search or find.
            8. The questions in the final list should be ranked from the most to the least formal suggestion.
            9. Each question should be written on a new line and the questions should be numbered from 1 (most formal) to 10 (least formal).
            10. The output should ONLY be the list of variations, and should not contain any other text."""


# ============= LANGCHAIN VERSION =============
def define_prompts_langchain(question):    
    messages = [
        ("system", system_prompt),
        ("human", question),
    ]
    return messages

def run_question_langchain(question):
    messages = define_prompts_langchain(question)
    ai_msg = llm.invoke(messages)
    print(f"Reference question: {question} \n")
    print("Generated variations:\n")
    print(ai_msg.content)
    return ai_msg

# ============= VERTEX AI NATIVE VERSION =============
def define_prompts_vertexai(question):    
    prompt = f"""{system_prompt}

Question: {question}"""
    return prompt

def run_question_vertexai(question):
    prompt = define_prompts_vertexai(question)
    ai_msg = llm.generate_content(prompt)
    print(f"Reference question: {question} \n")
    print("Generated variations:\n")
    print(ai_msg.text)
    return ai_msg

# ============= UNIFIED INTERFACE =============
# This function automatically selects the right version based on USE_LANGCHAIN flag
def run_question(question):
    if USE_LANGCHAIN:
        return run_question_langchain(question)
    else:
        return run_question_vertexai(question)



## Run generation

In [11]:
question_variations_dict = {}

for question in reference_questions:
    ai_msg = run_question(question)
    print('\n')
    
    # Extract text based on LLM type
    if USE_LANGCHAIN:
        response_text = ai_msg.content
    else:
        response_text = ai_msg.text
    
    qs = response_text.split('\n')
    formatted_variations = [q.split('. ', 1)[1] for q in qs if q.strip()]
    question_variations_dict[question] = formatted_variations

Reference question: What is the SMILES of {{molecule_name}}? 

Generated variations:

1. Could you please provide the SMILES representation for {{molecule_name}}?
2. I need to find the SMILES string for {{molecule_name}}, can you help with that?
3. Hey, can you pull up the SMILES for {{molecule_name}} for me?
4. wats the smiles for {{molecule_name}}?


Reference question: What is the SMARTS of {{molecule_name}}? 

Generated variations:

1. Could you please provide the SMARTS representation for {{molecule_name}}?
2. I need to find the SMARTS string for {{molecule_name}}, can you assist with that?
3. Hey, can you pull the SMARTS pattern for {{molecule_name}} for me?
4. whats the smarts for {{molecule_name}}??


Reference question: Can you generate similar molecules to {{smiles}}? 

Generated variations:

1. Could you please generate a set of molecules that are structurally similar to {{smiles}}?
2. Can you generate some analogs for me based on this molecule, {{smiles}}?
3. Hey, I need to

## Extend reference questions with the generated variations

In [ ]:
new_df = pd.DataFrame()

for ref in question_variations_dict.keys():
    df = reference_questions_df.loc[reference_questions_df['reference_question'] == ref]
    #display(df)
    df[['reference_question', 'task_type', 'task_label', 'tool_calls_complete', 'tool_calls_accepted']].drop_duplicates()
    #Add row for the reference question with variation_label = 0, and the same values for the other columns as the original question
    new_row = df.iloc[0].copy()
    new_df = pd.concat([new_df, pd.DataFrame([new_row])], ignore_index=True)  # Append the new row to the dataframe
    
    # Create multiple rows, one for each variation of the question, and keep the same values for the other columns as the original question
    for i, variation in enumerate(question_variations_dict[ref]):
        new_row = df.iloc[0].copy()  # Copy the first row of the dataframe
        new_row['variation_question'] = variation  # Update the reference question to the variation
        new_row['variation_label'] = i + 1  # Update the variation number
        new_df = pd.concat([new_df, pd.DataFrame([new_row])], ignore_index=True)  # Append the new row to the dataframe

display(new_df)

,reference_question,variation_label,variation_question,task_type,task_label,tool_calls_complete,tool_calls_accepted,note
0,What is the SMILES of {{molecule_name}}?,0,What is the SMILES of {{molecule_name}}?,Tool,Synonym2SMILES,synonyms2smiles,NaN,Question is well within the scope of the agent...
1,What is the SMILES of {{molecule_name}}?,1,Could you please provide the SMILES representa...,Tool,Synonym2SMILES,synonyms2smiles,NaN,Question is well within the scope of the agent...
2,What is the SMILES of {{molecule_name}}?,2,I need to find the SMILES string for {{molecul...,Tool,Synonym2SMILES,synonyms2smiles,NaN,Question is well within the scope of the agent...
3,What is the SMILES of {{molecule_name}}?,3,"Hey, can you pull up the SMILES for {{molecule...",Tool,Synonym2SMILES,synonyms2smiles,NaN,Question is well within the scope of the agent...
4,What is the SMILES of {{molecule_name}}?,4,wats the smiles for {{molecule_name}}?,Tool,Synonym2SMILES,synonyms2smiles,NaN,Question is well within the scope of the agent...
...,...,...,...,...,...,...,...,...
95,Can you give me some molecules similar to {{mo...,0,Can you give me some molecules similar to {{mo...,Workflow,DesignPropertySynthesis,synonyms2smiles;molformer;reinvent_scoring;Ana...,synonyms2smiles;molformer;reinvent_scoring;aiz...,Question is well within the scope of the agent...
96,Can you give me some molecules similar to {{mo...,1,Could you please generate a set of analogues f...,Workflow,DesignPropertySynthesis,synonyms2smiles;molformer;reinvent_scoring;Ana...,synonyms2smiles;molformer;reinvent_scoring;aiz...,Question is well within the scope of the agent...
97,Can you give me some molecules similar to {{mo...,2,I need you to suggest some molecules similar t...,Workflow,DesignPropertySynthesis,synonyms2smiles;molformer;reinvent_scoring;Ana...,synonyms2smiles;molformer;reinvent_scoring;aiz...,Question is well within the scope of the agent...
98,Can you give me some molecules similar to {{mo...,3,"Hey, can you pull up some analogues of {{molec...",Workflow,DesignPropertySynthesis,synonyms2smiles;molformer;reinvent_scoring;Ana...,synonyms2smiles;molformer;reinvent_scoring;aiz...,Question is well within the scope of the agent...


## Save the reference and generated test questions to file

In [ ]:
save_path = "variations_test_questions_generated.csv"

new_df.to_csv(save_path)